# Lesson 7: Line Following

Your mission: follow a dark line using sensor feedback. Start with steering, then test a straight path and gentle curves.

**Before you begin:** Use the Robot Car kernel. Verify grayscale calibration for this track in Betabox Calibration, then leave that tool. Keep the motors stopped while positioning the car. All movement tests require instructor approval, a safe floor area and an accessible power switch. Test new code on a secure stand first. Keep the sensors at normal track height using a securely positioned test sample, and never reach toward moving wheels. If readings cannot be reproduced safely on the stand, ask your instructor for help.

Run one activity at a time. The hardware activities are disabled until you deliberately enable them. Do not use Run All with hardware flags enabled.

## 1. Prepare your tools

Run this cell first. `read_line()` uses the calibrated API from Lesson 6. It returns `None` if usable data is unavailable. The cleanup helper stops, centers and closes each car session. The motion helper allows only a brief, low-power forward step.

In [ ]:
import math
import time
from betabox_robotics import BetaboxCar
from betabox_robotics.runtime.errors import RobotRuntimeError


def read_line(car, threshold=0.5):
    try:
        reading = car.line_reading(threshold=threshold)
    except RobotRuntimeError as error:
        print("Sensor error:", error)
        return None

    if reading.normalized is None:
        print("Grayscale calibration is missing.")
        return None

    if len(reading.normalized) != 3 or not all(
        math.isfinite(value) and 0 <= value <= 1
        for value in reading.normalized
    ):
        print("Invalid normalized readings.")
        return None

    status = list(reading.status)
    if len(status) != 3 or any(value not in (0, 1) for value in status):
        print("Invalid sensor pattern.")
        return None
    return status


def finish_driving(car):
    try:
        car.stop()
        car.center()
    finally:
        car.close()


def apply_steering(car, angle):
    # Negative = left, positive = right, zero = center.
    if angle is None:
        car.stop()
        car.center()
    elif not -30 <= angle <= 30:
        raise ValueError("Steering must be between -30 and 30 degrees.")
    elif angle < 0:
        car.left(abs(angle))
    elif angle > 0:
        car.right(angle)
    else:
        car.center()


def forward_step(car, speed=15, duration=0.1):
    if not 0 < speed <= 20 or not 0 < duration <= 0.2:
        raise ValueError("Use approved low power and a short movement step.")
    try:
        car.forward(speed)
        time.sleep(duration)
    finally:
        car.stop()

## 2. Plan and test your steering decisions

Complete `choose_angle()` using `if`, `elif` and `else`. Return `-15` for left only, `15` for right only, and `0` for center only. Return `None` for a lost line, multiple detections or invalid readings.

The sign represents a direction in our program, not a new car command. `apply_steering()` translates the angle into familiar car methods. Until you complete the function, it returns `None` for every reading, so it requests no movement.

In [ ]:
def choose_angle(status):
    # Your code here. Handle None before unpacking the three readings.
    return None

Run the tests below. Fix your function until all nine results match. This cell does not use the car.

In [ ]:
patterns = [
    ([0, 1, 0], 0),
    ([1, 0, 0], -15),
    ([0, 0, 1], 15),
    ([0, 0, 0], None),
    ([1, 1, 0], None),
    ([0, 1, 1], None),
    ([1, 0, 1], None),
    ([1, 1, 1], None),
    (None, None),
]

for status, expected in patterns:
    result = choose_angle(status)
    print(status, "Result:", result, "Expected:", expected,
          "PASS" if result == expected else "TRY AGAIN")

## 3. Test steering on the real car

Keep the drive motors stopped. Position the line under one sensor at a time, then off the line and under two sensors. Enable the cell only after your simulated tests pass. Observe the steering during the one-second pause. The wheels center when the session ends. Keep hands clear and reposition only when the session is over.

| Position | Expected steering | Actual result |
|---|---|---|
| Center | Center | |
| Left | Left | |
| Right | Right | |
| No line | Stop and center | |
| Two sensors | Stop and center | |

In [ ]:
run_steering_test = False

if run_steering_test:
    car = BetaboxCar()
    try:
        car.stop()
        status = read_line(car)
        angle = choose_angle(status)
        print("Pattern:", status, "Decision:", angle)
        apply_steering(car, angle)
        time.sleep(1)
    finally:
        finish_driving(car)
else:
    print("Steering test disabled.")

## 4. Build a straight-line follower

Complete the missing decision inside `follow_line()`. The supplied loop checks the time and limits the number of steps. The car is stopped before every sensor read. Your task is to call the supplied decision function, stop the loop when it returns `None`, and otherwise apply the steering angle and take a short forward step.

Use `break` for the stop decision. To respect the time budget, use the supplied `remaining` value to limit the step duration. A deadline limits the loop, but cannot interrupt a blocking hardware call or guarantee immediate physical stopping.

In [ ]:
def follow_line(car, decision, threshold=0.5, speed=15,
                duration=0.1, max_steps=30, max_seconds=12):
    deadline = time.monotonic() + max_seconds
    car.stop()

    for step in range(max_steps):
        if time.monotonic() >= deadline:
            print("Time limit reached.")
            break

        status = read_line(car, threshold)
        # Your code here: call decision(status) and store its result.
        angle = None
        print("Step:", step + 1, "Pattern:", status, "Angle:", angle)
        if status is None or angle is None:
            print("Stopped: no usable steering decision.")
            break

        # Your code here: apply the steering decision.
        remaining = deadline - time.monotonic()
        if remaining <= 0:
            print("Time limit reached.")
            break
        # Your code here: call forward_step with the speed and
        # a duration of min(duration, remaining).
    else:
        print("Check limit reached.")

    car.stop()

### Test 1: Wheels elevated

With instructor approval, enable the next cell and test on the stand. Confirm it stops for a lost line, ambiguous pattern, failed reading or check limit. Use secure test samples arranged before the run; do not reach under a running car. Start another test after the previous session has closed.

### Test 2: Straight path

After stand testing, place the car at the start of the straight line on the floor. Use the same cell. Supervise the test and stop it if the car leaves the marked area. Adjust only one setting at a time. Keep the step short and motor power low.

In [ ]:
run_straight_test = False

if run_straight_test:
    car = BetaboxCar()
    try:
        follow_line(car, choose_angle)
    finally:
        finish_driving(car)
else:
    print("Straight-line test disabled.")

## 5. Final project: Follow a curved track

Create an improved decision function. Keep your original single-sensor rules. Add a smaller left correction for `[1, 1, 0]` and a smaller right correction for `[0, 1, 1]`. Start with 8 degrees for the smaller corrections. Stop for `[0, 0, 0]`, `[1, 0, 1]`, `[1, 1, 1]` or missing readings.

Reuse your completed `follow_line()` rather than rewriting the driving loop. First pass every simulated test. Then repeat stand testing before attempting a track with a straight section and gentle left and right curves.

In [ ]:
def choose_curve_angle(status):
    # Your code here.
    return None

curve_patterns = [
    ([0, 1, 0], 0),
    ([1, 0, 0], -15),
    ([0, 0, 1], 15),
    ([1, 1, 0], -8),
    ([0, 1, 1], 8),
    ([0, 0, 0], None),
    ([1, 0, 1], None),
    ([1, 1, 1], None),
    (None, None),
]

for status, expected in curve_patterns:
    result = choose_curve_angle(status)
    print(status, "Result:", result, "Expected:", expected,
          "PASS" if result == expected else "TRY AGAIN")

In [ ]:
run_curve_test = False

if run_curve_test:
    car = BetaboxCar()
    try:
        follow_line(car, choose_curve_angle, threshold=0.5,
                    speed=15, duration=0.1, max_steps=30, max_seconds=12)
    finally:
        finish_driving(car)
else:
    print("Curved-track test disabled.")

## 6. Test and improve

Reset the starting position for each attempt. Record one change and the result. Keep the test within instructor-approved bounds.

| Attempt | Setting changed | What happened? | Next adjustment |
|---|---|---|---|
| 1 | Baseline | | |
| 2 | | | |
| 3 | | | |

Demonstrate a straight section, a gentle left curve and a gentle right curve. Confirm that a lost line or ambiguous reading stops the robot and that the run ends at its check or time limit. Your instructor may allow repeated bounded tests to complete a longer course.

**Extra challenge:** Count the steering corrections during a run, or experiment with smaller correction angles to reduce zigzagging.

## Return to the portal

Confirm the car is stopped and every hardware session has closed. If the car behaves unexpectedly, use the instructor-approved physical stop procedure. Set all three hardware test flags back to `False`, save your notebook, and return to the Lesson 7 webpage to reflect on your results.